# Day2-Cross-Validation
By the time you reach the end of this notebook,you will be able to :
- State why using cross-validation gives us a better measure of performance than using one validation split.
- Describe how 5-fold corss validation works.
- Use cross val_score to test the performance of machine learning models.
- Compute the mean and standard deviation of the cross-validation scores.
- State the importance of Stratified k-fold in classification tasks.
- Compare cross validation performance to one train/test split.

## Introdution 
Cross-Validation is one of the techniques which are applied to evaluate machine learning models,it does not consider only one training and test splits of the data but it uses k-folds cross-validation when the dataset is divded into k-folds.
In total the process of training and validation is repeated k-times .In each iteration:
- one fold is left for validation.
- other k-1 folds are used for training.
- every sample is involved in validation just once.
The cross-validation estimate is usually obtained by taking the mean of the scores across the k folds.Thus,the evaluation proccess becomes more reliable as the result does not depend on just one random split of the dataset.

# Lesson Content (Training)

## Import Libraries

In [73]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import(train_test_split,StratifiedKFold)
from sklearn.metrics import(f1_score,accuracy_score,classification_report,confusion_matrix)
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression


## Load the Dataset
We will use the same student_performance dataset used in day1.
The target was Final Exam score but in todays classification task we will create a new target called **Pass**.
The Pass target will tell us whether the student's final exam score is less than 60 or not.

In [5]:
df=pd.read_csv("student_performance_dataset.csv")
df.head(10)

,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,1,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A
1,2,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A
2,3,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A
3,4,Male,2.6,77.5,8.0,NaN,Yes,Yes,No,85.1,83.8,B
4,5,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D
5,6,Female,4.2,78.2,5.7,High School,Yes,No,No,79.8,81.6,B
6,7,Male,1.5,100.0,5.0,High School,Yes,Yes,Yes,58.6,69.6,D
7,8,Male,6.2,86.4,6.0,High School,Yes,Yes,No,78.4,88.0,B
8,9,Male,5.3,81.3,6.7,Bachelors,Yes,No,No,63.8,84.4,B
9,10,Female,2.8,86.8,5.1,Bachelors,Yes,Yes,No,57.7,74.3,C


## check the dataset
This will help us understand the strucute of the dataset befor training the model begin.

In [ ]:
print("Dataset Shape is :",df.shape)
print("Number of null values:")
print(df.isnull().sum())
df["parental_education"]=df["parental_education"].fillna('Unkwon')

Dataset Shape is : (1000, 12)
Number of null values:
student_id                    0
gender                        0
study_time_hours              0
attendance_percent            0
sleep_hours                   0
parental_education            0
internet_access               0
extracurricular_activities    0
part_time_job                 0
previous_grade                0
final_exam_score              0
final_grade                   0
dtype: int64


## Define the Classification Target
The original dataset consists of Final Exam Score that is consists numeric variable ,for the classification tasks today it is transformed into two classes:
- 0=Fail
- 1=Pass
Score greater than or equal to 60 are classified as pass.
we will remove this two columns from the feature x :
- Final Exam Score
- Final Grade
This prevents the model form receiving information that directly reveals the target.

In [17]:
df["Pass"]=(df["final_exam_score"]>=60).astype(int)
df["Pass"].value_counts()

Pass
1    988
0     12
Name: count, dtype: int64

The above output shows the number of students in each category  
- 0 number of failed students
- 1 numbet of passed students
The significance of the information comes from the fact that classificaiton datasets may be unbalanced as this dataset,this happen when one category is larget compared to other ,so it becomes necessary to ensure that the distribution of classes is preserved in the validation fold.


## Define X and Y
The target variable (y) is pass and the feature matrix x contains the avaliable student information.

In [18]:
x=df.drop(columns=["final_exam_score","final_grade","Pass"])
y=df["Pass"]
print("X shape is",x.shape)
print("Y shape is",y.shape)

X shape is (1000, 10)
Y shape is (1000,)


## Identify Numerical and Categorical Features
Machine Learning requires numerical inputs ,so we should seprate the columns into 
- Numerical columns
- Categorical columns
Categorical will be convertied into numerical values using (**One-Hot-Encoding**)

In [58]:
num_col=x.select_dtypes(include=["int64","float64"]).columns.tolist()
cat_col=x.select_dtypes(include=["object","category","bool"]).columns.tolist()


## Preprocessing
Categorical features are converted into numerical values using one-hot encoding before training the model.

In [ ]:
x = pd.get_dummies(x, columns=cat_col, drop_first=True)

## Train/Test Split
The test set will remain the same (untouched) during the corss-validation 
we use:
- 80% for training
- 20% for final testing

In [60]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

## Evaluating a Decision Tree with 5-Fold Cross-Validation
For the evaluation of the model we will use the Decision Tree Classifier,the model is being evaluated using 5-fold cross-validation 
each time :
- 4 folds for training
- 1 folds if for testing
it is done for 5 times ,we will use F1 score as the evalution metric since it is a classification problem and F1-score is considers both precision and recall.

In [61]:
model = DecisionTreeClassifier(max_depth=3, random_state=42)
scores = cross_val_score(model, x_train, y_train, cv=5, scoring="f1")
print("F1 score for each fold is")
print(scores)


F1 score for each fold is
[0.99371069 0.99371069 0.99371069 0.99053628 0.99367089]


In [62]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(kf.split(x_train), 1):
    print(f"Fold {fold}")
    print("Training samples:", len(train_idx))
    print("Validation samples:", len(val_idx))
    print()

Fold 1
Training samples: 640
Validation samples: 160

Fold 2
Training samples: 640
Validation samples: 160

Fold 3
Training samples: 640
Validation samples: 160

Fold 4
Training samples: 640
Validation samples: 160

Fold 5
Training samples: 640
Validation samples: 160



Each sample is used for validation once and for training four times.

## Results Analysis
Since the dataset is extremely imbalanced, there are 988 students who pass the test vs only 12 students who fail
The F1-scores of the folds are:
- **Fold 1:** 0.99371069
- **Fold 2:** 0.99371069
- **Fold 3:** 0.99371069
- **Fold 4:** 0.99053628
- **Fold 5:** 0.99367089
Class Imbalance Handling: The model could still maintain a decent performance metric despite the extremely unbalanced dataset due to the use of stratification and class weighting.
*Model Consistency: The F1-scores did not show significant change across all of the folds (~0.9905 vs 0.9937).

## Calculate the mean and Standard Deviation
Mean refers to the performance of the model on an average basis for all the five folds.
The standard Deviation is a measure of variation in performance from fold to fold.
Having a high mean and low standard devition is normally a good combination since it indicates:
- Performance of the model is good.
- Performance is consiste.

In [63]:
mean=scores.mean()
std=scores.std()
print(f"Mean f1-score is:{mean}")
print(f"Standard Deviation  is:{std}")



Mean f1-score is:0.9930678478300342
Standard Deviation  is:0.001265878994548848


## Results Analysis
Mean F1 Score:0.9931 (suggesting an extremely high mean performance across all the folds).
Standard Deviation: 0.0013 (showing extremely low variation in the performance of the model from fold to fold).

## Stratified K-fold
In terms of classifiying problems it is desired that each fold would have an equal representation of each class.
For instance our dataset is 
1    988
0     12
then each fold should have approximately the same proportion,Stratified K-fold maintains the class distribution among the folds.
This is particulary true when the dataset is imbalanced.

In [64]:
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

When cv=5 is used with a classifier, scikit-learn uses StratifiedKFold by default.

In [65]:
m=DecisionTreeClassifier(max_depth=3,random_state=42)
score=cross_val_score(m,x_train,y_train,cv=skf,scoring="f1")
print("F1 score for each fold is:",score)
print("Mean F1-score is:",score.mean())

F1 score for each fold is: [0.99053628 0.99684543 0.99371069 0.99053628 0.99053628]
Mean F1-score is: 0.9924329900997957


## Why Stratified K-fold is matters in This case
### 1. Validation:
Stratified 5-Fold Cross-Validation (StratifiedKFold) has been specifically defined and included in the cross_val_score method by cv=skf.
### 2. Reasons for Stratification:
- Extreme Class Imbalance: The dataset is highly skewed because there are **988 passing students (Class 1)** while there are only **12 failing students (Class 0)**.
- Equal Class Proportion in Splits: Normal K-Fold splitting involves random data partitioning, which means there is a high probability that some folds will have very few or no instances of minority classes (failing students).
- Equal Evaluation: By using StratifiedKFold, each and every fold will have the exact proportion of passing students and failing students (about 98.8% to 1.2%).

# Hands-On Lab
## Step 1: Take a Week 3 model and evaluate it with 5-fold cross-validation using cross_val_score.

In [66]:
model = LogisticRegression(max_iter=1000, random_state=42)
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,x_train,y_train,cv=5,scoring="f1")
print("F1 score for each fold:")
print(scores)

F1 score for each fold:
[0.99367089 0.99371069 0.99371069 0.99371069 0.99684543]


The following are the F1-scores from the cross-validation process:
Fold 1: 0.99367089
Fold 2: 0.99371069
Fold 3: 0.99371069
Fold 4: 0.99371069
Fold 5: 0.99684543
High Stability: There is great consistency in the scores, where the average F1-score ranges around 0.9937 with some improvement at the end.
Ready for Model Comparison: This is because the performance of the model using the standard K-Fold approach has been established as a benchmark.

## Step 2: Report the mean and standard deviation of the scores across folds.

In [67]:
mean = scores.mean()
std = scores.std()
print("mean F1-score:", mean)
print("Standard Deviation:", std)

mean F1-score: 0.994329677483031
Standard Deviation: 0.0012579686634153215


Mean F1-score is the measure of average performance of the model on the five folds,the standard deviation is an indicator of the variability in the performance of the model across the folds,high mean and low standard deviation imply good and consistent performance of the model.

## Step 3: Compare the cross-validated estimate to the single-split score from Day 1 and explain any difference.
### First train the Model on the Single Training Split

In [68]:
model.fit(x_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

### Predict on the Single Test Split

In [69]:
y_predict= model.predict(x_test)
single = f1_score(y_test, y_predict)
print("Single-split F1-score:", single)

Single-split F1-score: 0.9949748743718593


###  Compare the Two Scores

In [70]:
print("Single-split F1-score:", single_split_score)
print("Cross-validation mean F1-score:", mean_score)

Single-split F1-score: 0.9949748743718593
Cross-validation mean F1-score: 0.994329677483031


The single-split score only relies on a single train/test split,while the cross-validation score relies on the five different folds,this implies that the cross-validation mean score represents a more reliable estimate of the model's performance since it is not dependent on just one train/test split.

## Step 4: For a classification task, confirm stratified folds are used and explain why that matters here.

In [71]:
for fold, (train_idx, val_idx) in enumerate(skf.split(x_train, y_train), 1):
    print(f"Fold {fold}")
    print(y_train.iloc[val_idx].value_counts())
    print()

Fold 1
Pass
1    158
0      2
Name: count, dtype: int64

Fold 2
Pass
1    158
0      2
Name: count, dtype: int64

Fold 3
Pass
1    158
0      2
Name: count, dtype: int64

Fold 4
Pass
1    158
0      2
Name: count, dtype: int64

Fold 5
Pass
1    158
0      2
Name: count, dtype: int64



### Use StratifiedKFold with cross_val_score

In [72]:
stratified_scores = cross_val_score(model,x_train,y_train,cv=skf,scoring="f1")
print("F1 scores using Stratified K-Fold:")
print(stratified_scores)
print("Mean F1-score:", stratified_scores.mean())
print("Standard deviation:", stratified_scores.std())

F1 scores using Stratified K-Fold:
[0.99684543 0.99684543 0.99371069 0.99371069 0.99371069]
Mean F1-score: 0.9949645854413429
Standard deviation: 0.0015356997772344677


The stratified K-Fold method keeps the proportion of all classes in each fold constant.
It is particularly significant in imbalanced classification problems, where both classes must be present in the validation folds.
Otherwise, the minority class may not have enough representation in some of the folds.